# Markov chains and PageRank

*maths · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/markov_pagerank)

A random surfer clicks links on a small synthetic web and now and then jumps to a random page. The probability of being on each page evolves by one matrix–vector product per click, and its stationary distribution is PageRank, found here by power iteration, by an eigen-decomposition and by thousands of simulated surfers at once.

**Vector form**

$$G = d\,S + \frac{1-d}{N}\,\mathbf 1\mathbf 1^T,\qquad \mathbf x_{k+1} = G\,\mathbf x_k,\qquad G\boldsymbol\pi = \boldsymbol\pi,\quad \mathbf 1^T\boldsymbol\pi = 1$$

**Index (tensor) form**

$$d_j = \sum_i L_{ij},\quad S_{ij} = L_{ij}/d_j,\qquad G_{bij} = d_b\,S_{ij} + \frac{1 - d_b}{N},\qquad x_{bi} \leftarrow G_{bij}\,x_{bj}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $L_{ij}$ | `web_graph(n, n_dangling, seed, device)` | (N, N) | 1 if page $j$ links to page $i$: columns are the pages a surfer leaves |
| $d_j = \sum_i L_{ij}$ | `einsum('ij->j', L)` | (N,) | summed over $i$: number of outgoing links of page $j$ |
| $S_{ij} = L_{ij}/d_j$ | `transition(L)` | (N, N) | click a random link; a dangling page ($d_j = 0$) gets the uniform column $1/N$ |
| $G_{bij}$ | `d[:, None, None] * S + (1 - d[:, None, None]) / N` | (B, N, N) | Google matrix for a batch of damping factors $d_b$; $1 - d$ is the teleport probability |
| $G_{bij}\,x_{bj}$ | `einsum('bij,bj->bi', G, x)` | (B, N) | repeated $j$ ⇒ summed: one power-iteration step for every $d_b$ at once |
| $G\boldsymbol\pi = \boldsymbol\pi$ | `torch.linalg.eig(G)` | (N,), (N, N) | eigenvector of eigenvalue 1, scaled to sum to 1; $G$ is not symmetric, so eig rather than eigh |
| $\vert \lambda_2\vert $ | `vals.abs().sort(descending=True)[1]` | () | second-largest eigenvalue modulus: the error shrinks by this factor per step, and $/\lambda_2/ \le d$ |
| $w_t$ | `walk_step(pos, S, d, gen)` | (W,) | page of each of $W$ surfers: follow a link with probability $d$, else teleport |

$S$ is column-stochastic, so it maps probability vectors to probability vectors, but a web graph has pages with no links out and groups of pages that trap the surfer. Dangling pages are given a uniform column (the surfer types a random address), and the teleport term makes every entry of $G$ positive. By the Perron–Frobenius theorem the stationary distribution is then unique, positive and reached from any start. Every other eigenvalue of $G$ is $d$ times an eigenvalue of $S$, so $|\lambda_2| \le d$ and power iteration converges at least as fast as $d^k$. The right panel shows the batch of damping factors solved together. With $d = 0.85$ (the value suggested by Brin and Page, 1998) 50 steps shrink the error by at least $0.85^{50} \approx 3\times10^{-4}$, whatever the size of the web. The surfers are a Monte Carlo estimate of the same chain: their fractions follow $\mathbf x_k$ up to noise of order $1/\sqrt W$, and averaging over time gives ranks that agree with power iteration. Removing the dangling fix lets probability leak away, as the terminal shows. Try $d \to 1$ to see convergence slow down and the ranks pile up in closed groups of pages.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/markov_pagerank/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import google, transition


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    d, n_iter = params["damping"], params["iterations"]
    walkers = 2000 if fast_mode() else params["walkers"]
    links, community, pos = tensors.web_graph(params["pages"], params["dangling"], params["seed"], dev)
    links = links.double()
    n = links.shape[0]
    s = transition(links)  # (N, N), column-stochastic
    print(f"web graph: {n} pages, {int(links.sum())} links, {int((links.sum(0) == 0).sum())} dangling pages")
    print(f"columns of S sum to 1: max |Σ_i S_ij − 1| = {(s.sum(0) - 1).abs().max().item():.1e}")

    ds = sorted({0.5, 0.7, 0.95, d})  # a batch of damping factors, solved together
    main = ds.index(d)
    g = google(s, torch.tensor(ds, dtype=s.dtype, device=dev))  # (B, N, N)
    history, residual = solve.power_iteration(g, n_iter)  # (K+1, B, N), (K, B)
    factor = solve.convergence_factor(residual)
    print("damping d   |λ₂| of G   observed rate ‖x_k+1 − x_k‖ / ‖x_k − x_k−1‖")
    lam2 = torch.stack([solve.eig_stationary(g[b])[1][1] for b in range(len(ds))])  # (B,)
    for b, db in enumerate(ds):
        print(f"   {db:.2f}      {lam2[b].item():.4f}      {factor[b].item():.4f}")

    pi = history[-1, main]
    pi_eig, _ = solve.eig_stationary(g[main])
    print(
        f"d = {d}: power iteration after {n_iter} steps vs eig: max difference {(pi - pi_eig).abs().max().item():.1e}"
    )
    print(f"PageRank sums to {pi.sum().item():.12f}, smallest entry {pi.min().item():.4f} (≥ (1 − d)/N)")

    naive = d * links / links.sum(0).clamp(min=1) + (1 - d) / n  # dangling columns left as zeros
    x = torch.full((n,), 1.0 / n, dtype=s.dtype, device=dev)
    for _ in range(n_iter):
        x = naive @ x
    print(f"without the dangling fix the total probability leaks away: Σ x = {x.sum().item():.4f} after {n_iter} steps")

    occupancy = solve.monte_carlo(s, d, walkers, n_iter, params["seed"])  # (K+1, N)
    estimate = occupancy[n_iter // 2 :].mean(dim=0).to(pi)  # time average once the surfers have mixed
    pearson = torch.corrcoef(torch.stack([estimate, pi]))[0, 1].item()
    print(f"{walkers} random surfers: L1 distance to PageRank {(estimate - pi).abs().sum().item():.4f}, ", end="")
    print(f"correlation {pearson:.4f}, rank correlation {solve.spearman(estimate, pi):.4f}")
    top = pi.argsort(descending=True)[:5].tolist()
    print("top pages (power):  " + "  ".join(f"#{i} {pi[i].item():.3f}" for i in top))
    top_mc = estimate.argsort(descending=True)[:5].tolist()
    print("top pages (surfers):" + "  ".join(f"#{i} {estimate[i].item():.3f}" for i in top_mc))

    return plot.animate(
        links,
        community,
        pos,
        history[:, main],
        pi_eig,
        occupancy,
        residual,
        ds,
        factor,
        lam2,
        d,
        out_dir / "pagerank.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/markov_pagerank/equation.py
r"""Markov chain on a web graph → tensor translation.

A random surfer on page j clicks one of its d_j = \sum_i L_{ij} outgoing links at random (i summed), so the
transition matrix is column-stochastic,

    S_{ij} = L_{ij} / d_j,       and S_{ij} = 1/N for a dangling page (d_j = 0: jump anywhere).

With probability 1 − d the surfer instead teleports to a uniformly random page (Google matrix),

    G_{ij} = d\,S_{ij} + (1 - d)/N,

and the probability of being on each page evolves as x_i \leftarrow G_{ij} x_j (j summed). PageRank is the
stationary distribution: G_{ij} \pi_j = \pi_i, \sum_i \pi_i = 1, the eigenvector with eigenvalue 1.
"""

import torch


def transition(links: torch.Tensor) -> torch.Tensor:
    r"""S_{ij} = L_{ij} / d_j with d_j = Σ_i L_{ij}; dangling columns become uniform.   (N, N) → (N, N)"""
    n = links.shape[0]
    out_degree = torch.einsum("ij->j", links)  # (N,)
    s = links / out_degree.clamp(min=1)
    return torch.where(out_degree == 0, 1.0 / n, s)  # broadcasts the (N,) mask along the columns


def google(s: torch.Tensor, d: torch.Tensor) -> torch.Tensor:
    r"""G_{bij} = d_b S_{ij} + (1 − d_b)/N for a batch of damping factors d_b.   (N, N), (B,) → (B, N, N)"""
    n = s.shape[0]
    return d[:, None, None] * s + (1 - d[:, None, None]) / n


def power_step(g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'_{bi} = G_{bij} x_{bj}   (j summed).   (B, N, N), (B, N) → (B, N)"""
    return torch.einsum("bij,bj->bi", g, x)


def walk_step(pos: torch.Tensor, s: torch.Tensor, d: float, gen: torch.Generator) -> torch.Tensor:
    """Move W surfers at once: follow a link (column pos of S) with probability d, else teleport.   (W,) → (W,)"""
    n = s.shape[0]
    follow = torch.multinomial(s.T[pos], 1, generator=gen)[:, 0]  # (W,): row w of S.T[pos] is page pos_w's links
    jump = torch.randint(0, n, pos.shape, generator=gen)
    teleport = torch.rand(pos.shape, generator=gen) > d
    return torch.where(teleport, jump, follow)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/markov_pagerank/tensors.py
"""Build the concrete tensors: a small synthetic web graph (links, communities, dangling pages, drawing positions)."""

import math

import torch


def web_graph(n: int, n_dangling: int, seed: int, device: torch.device):
    """A random directed graph of n pages in 3 communities, with a few famous pages and n_dangling dead ends.

    Returns L_{ij} (n, n) with L_{ij} = 1 if page j links to page i (columns are sources), the community of
    every page (n,), and 2D positions for drawing (n, 2).
    """
    gen = torch.Generator().manual_seed(seed)
    community = torch.arange(n) % 3  # (n,)
    fame = 1.0 / (1.0 + torch.randperm(n, generator=gen).float()) ** 0.8  # a few pages are much more popular
    links = torch.zeros(n, n)
    dangling = torch.randperm(n, generator=gen)[:n_dangling]  # pages with no outgoing links
    for j in range(n):
        if j in dangling:
            continue
        weight = fame * torch.where(community == community[j], 4.0, 1.0)  # prefer famous pages nearby
        weight[j] = 0.0  # no self-links
        n_out = int(torch.randint(1, 5, (1,), generator=gen))
        targets = torch.multinomial(weight, n_out, replacement=False, generator=gen)
        links[targets, j] = 1.0

    angle = 2 * math.pi * community / 3 + 0.5  # communities sit around a circle
    centre = torch.stack([torch.cos(angle), torch.sin(angle)], dim=1)
    spread = 0.55 * torch.rand(n, generator=gen).sqrt()
    phi = 2 * math.pi * torch.rand(n, generator=gen)
    pos = centre + spread[:, None] * torch.stack([torch.cos(phi), torch.sin(phi)], dim=1)
    return links.to(device), community, pos


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/markov_pagerank/solve.py
"""Find the stationary distribution three ways: power iteration, an eigen-decomposition and many random surfers."""

import torch

from problems._common.progress import steps

from .equation import power_step, walk_step


def power_iteration(g: torch.Tensor, n_iter: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Start from the uniform distribution and apply G n_iter times, for every damping factor in the batch.

    g: (B, N, N). Returns the history x (n_iter+1, B, N) and the residuals ‖x_{k+1} − x_k‖₁ (n_iter, B).
    """
    b, n, _ = g.shape
    x = torch.full((b, n), 1.0 / n, dtype=g.dtype, device=g.device)
    history = [x]
    for _ in steps(n_iter, "power iterations"):
        x = power_step(g, x)
        history.append(x)
    history = torch.stack(history)
    residual = (history[1:] - history[:-1]).abs().sum(dim=-1)
    return history, residual


def eig_stationary(g: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """The eigenvector of G (N, N) for eigenvalue 1, scaled to sum to 1, and all |λ| in decreasing order.

    G is not symmetric, so this needs the general torch.linalg.eig (eigh is only for symmetric matrices).
    """
    vals, vecs = torch.linalg.eig(g.double())
    top = vals.real.argmax()
    pi = vecs[:, top].real
    return (pi / pi.sum()).to(g.dtype), vals.abs().sort(descending=True).values


def convergence_factor(residual: torch.Tensor, floor: float = 1e-12) -> torch.Tensor:
    """Observed rate r with residual_k ∝ r^k: median ratio of successive residuals above round-off.   (K, B) → (B,)"""
    out = []
    for res in residual.T:
        good = res[res > floor]
        ratios = good[1:] / good[:-1]
        out.append(ratios[len(ratios) // 2 :].median())  # the late ratios, after the start-up transient
    return torch.stack(out)


def monte_carlo(s: torch.Tensor, d: float, n_walkers: int, n_steps: int, seed: int) -> torch.Tensor:
    """Fraction of W surfers on each page after every step, starting uniformly: (n_steps+1, N).

    Walkers live on the CPU because the random generator does.
    """
    s = s.cpu()
    n = s.shape[0]
    gen = torch.Generator().manual_seed(seed)
    pos = torch.randint(0, n, (n_walkers,), generator=gen)  # (W,)
    occupancy = [torch.bincount(pos, minlength=n) / n_walkers]
    for _ in steps(n_steps, "surfer steps"):
        pos = walk_step(pos, s, d, gen)
        occupancy.append(torch.bincount(pos, minlength=n) / n_walkers)
    return torch.stack(occupancy)


def spearman(a: torch.Tensor, b: torch.Tensor) -> float:
    """Rank correlation of two score vectors (Pearson correlation of their ranks)."""
    ra, rb = a.argsort().argsort().double(), b.argsort().argsort().double()
    return torch.corrcoef(torch.stack([ra, rb]))[0, 1].item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/markov_pagerank/plot.py
"""Animate the probabilities flowing over the graph, the ranks from three methods, and the convergence rates."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(links, community, pos, history, pi_eig, occupancy, residual, ds, factor, lam2, d, path: Path) -> Path:
    """history (K+1, N) power iterates for damping d, occupancy (K+1, N) surfer fractions, residual (K, B) for ds."""
    lk, xy = links.cpu().numpy(), pos.cpu().numpy()
    x, mc, pi = history.cpu().numpy(), occupancy.cpu().numpy(), pi_eig.cpu().numpy()
    res, rate, lam = residual.cpu().numpy(), factor.cpu().numpy(), lam2.cpu().numpy()
    n, n_steps = len(pi), len(x)
    colours = np.array([ACCENT[c] for c in community.tolist()])
    dangling = lk.sum(axis=0) == 0

    fig, (ax_g, ax_b, ax_r) = plt.subplots(1, 3, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.3, 1]})
    for i, j in zip(*np.nonzero(lk)):  # an arrow for every link j → i
        ax_g.annotate(
            "",
            xy=xy[i],
            xytext=xy[j],
            arrowprops={"arrowstyle": "-|>", "color": "#9ca3af", "lw": 0.6, "shrinkA": 4, "shrinkB": 6},
        )
    nodes = ax_g.scatter(xy[:, 0], xy[:, 1], s=40, c=colours, zorder=3, edgecolors="#374151", linewidths=0.6)
    ax_g.scatter(*xy[dangling].T, s=90, facecolors="none", edgecolors="#dc2626", linewidths=1.5, zorder=4)
    ax_g.set_aspect("equal")
    ax_g.axis("off")
    ax_g.set_title("web graph (red rings: dangling pages)", fontsize=9)

    order = np.argsort(-pi)  # pages sorted by their final rank
    ranks = np.arange(n)
    bars = ax_b.bar(ranks, x[0, order], color=colours[order], alpha=0.75, label="power iteration $x_k$")
    (dots,) = ax_b.plot(ranks, mc[0, order], "o", color="#f59e0b", ms=4, label="surfer fractions")
    ax_b.plot(ranks, pi[order], "_", color="#111827", ms=9, mew=1.5, label="eigenvector of $G$")
    ax_b.set_xlabel("page, sorted by PageRank")
    ax_b.set_ylabel("probability")
    ax_b.set_ylim(0, 1.25 * max(pi.max(), mc.max()))
    ax_b.legend(fontsize=7, loc="upper right")

    k = np.arange(1, res.shape[0] + 1)
    for b, db in enumerate(ds):
        col = ACCENT[b % len(ACCENT)]
        ax_r.semilogy(
            k,
            res[:, b],
            color=col,
            lw=1.6 if db == d else 0.9,
            label=f"d = {db:.2f}: rate {rate[b]:.3f}, |λ₂| {lam[b]:.3f}",
        )
        ax_r.semilogy(k, res[0, b] * db ** (k - 1.0), "--", color=col, lw=0.7, alpha=0.6)
    ax_r.set_ylim(1e-15, 1)
    ax_r.set_xlabel("iteration $k$")
    ax_r.set_title(r"$\|x_{k+1} - x_k\|_1$ (dashed: $d^k$)", fontsize=9)
    ax_r.legend(fontsize=7, loc="lower left")
    marker = ax_r.axvline(0, color="#374151", lw=0.8)
    title = fig.suptitle("", fontsize=11)

    def update(t: int):
        nodes.set_sizes(25 + 2500 * x[t])
        for bar, h in zip(bars, x[t, order]):
            bar.set_height(h)
        dots.set_ydata(mc[t, order])
        marker.set_xdata([max(t, 1)])
        title.set_text(f"PageRank with damping d = {d}: step {t}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_steps, 120), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'damping': 0.85,
    'pages': 30,
    'dangling': 3,
    'walkers': 20000,
    'iterations': 60,
    'seed': 3
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.markov_pagerank")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.markov_pagerank.main")
problem_dir = Path("problems/maths/markov_pagerank")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
